In [ ]:
import subprocess
import sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'huggingface-hub>=0.26.0',
    'python-dotenv>=1.0.0',
    'pyyaml>=6.0',
    'requests>=2.32.0',
], check=True)

In [ ]:
import os
import json
import time
import threading
from pathlib import Path
from datetime import datetime, timezone

import yaml
import requests
from huggingface_hub import HfApi, CommitOperationAdd

WORK_DIR        = Path('/kaggle/working')
CLEAN_DIR       = WORK_DIR / 'clean_final'
METADATA_PATH   = WORK_DIR / 'metadata.jsonl'
CHECKPOINT_PATH = WORK_DIR / 'checkpoint_p1e.json'
CONFIG_DIR      = Path('/kaggle/input/datasets/mirza176528/s2s-pipline-v2-0-2/config')

WAVE_SIZE_BYTES = 500 * 1024 * 1024
BATCH_SIZE      = 50
MAX_RETRIES     = 12
COMMIT_DELAY    = 3.0

In [ ]:
def load_secrets():
    try:
        from kaggle_secrets import UserSecretsClient
        c = UserSecretsClient()
        secrets = {
            'HF_TOKEN_PRIMARY':   c.get_secret('HF_TOKEN_PRIMARY'),
            'HF_TOKEN_SECONDARY': c.get_secret('HF_TOKEN_SECONDARY'),
            'HF_TOKEN_TERTIARY':  c.get_secret('HF_TOKEN_TERTIARY'),
            'GEMINI_API_KEY':  c.get_secret('GEMINI_API_KEY_01') or c.get_secret('GEMINI_API_KEY'),
        }
        print('[secrets] loaded from Kaggle Secrets')
        return secrets
    except Exception:
        pass
    env_file = Path('.env')
    if env_file.exists():
        from dotenv import load_dotenv
        load_dotenv(env_file)
        print('[secrets] loaded from .env')
    required = ['HF_TOKEN_PRIMARY', 'HF_TOKEN_SECONDARY', 'HF_TOKEN_TERTIARY']
    missing = [k for k in required if not os.environ.get(k)]
    if missing:
        raise RuntimeError(f'Missing secrets: {missing}')
    return {k: os.environ[k] for k in required}

SECRETS     = load_secrets()
HF_TOKEN    = SECRETS['HF_TOKEN_PRIMARY']

with open(CONFIG_DIR / 'hf_repos.yaml') as f:
    repos_cfg = yaml.safe_load(f)

STAGE0_REPO     = repos_cfg['repos']['stage0_codec']['repo_id']
OVERFLOW_REPO   = repos_cfg['repos']['overflow']['repo_id']
OVERFLOW_TOKEN  = SECRETS['HF_TOKEN_TERTIARY']
HF_API          = HfApi(token=HF_TOKEN)
print(f'[config] primary repo : {STAGE0_REPO}')
print(f'[config] overflow repo: {OVERFLOW_REPO}')

In [ ]:
def load_checkpoint():
    if CHECKPOINT_PATH.exists():
        try:
            with open(CHECKPOINT_PATH) as f:
                state = json.load(f)
            print(f'[checkpoint] local — uploaded={state["stats"]["uploaded"]} waves={state["stats"]["waves_committed"]}')
            return state
        except Exception:
            pass
    try:
        url = f'https://huggingface.co/datasets/{STAGE0_REPO}/resolve/main/checkpoint_p1e.json'
        r = requests.get(url, headers={'Authorization': f'Bearer {HF_TOKEN}'}, timeout=30)
        if r.status_code == 200:
            state = r.json()
            with open(CHECKPOINT_PATH, 'w') as f:
                json.dump(state, f)
            print(f'[checkpoint] HF fallback — uploaded={state["stats"]["uploaded"]}')
            return state
    except Exception:
        pass
    print('[checkpoint] fresh start')
    return {
        'uploaded_ids': [],
        'failed_ids': [],
        'metadata_uploaded': False,
        'stats_uploaded': False,
        'stats': {
            'uploaded': 0,
            'failed': 0,
            'waves_committed': 0,
            'overflow_files': 0,
        },
        'last_updated': None,
    }


cp_lock = threading.Lock()

def save_checkpoint(state, upload=False):
    with cp_lock:
        state['last_updated'] = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')
        tmp = str(CHECKPOINT_PATH) + '.tmp'
        with open(tmp, 'w') as f:
            json.dump(state, f)
        os.replace(tmp, str(CHECKPOINT_PATH))
    if not upload:
        return
    for attempt in range(6):
        try:
            HF_API.upload_file(
                path_or_fileobj=json.dumps(state).encode(),
                path_in_repo='checkpoint_p1e.json',
                repo_id=STAGE0_REPO,
                repo_type='dataset',
                commit_message='p1e checkpoint',
            )
            return
        except Exception as e:
            time.sleep(min(2 ** attempt, 60))


state       = load_checkpoint()
uploaded_set = set(state['uploaded_ids'])

In [ ]:
def get_repo_size_gb(repo_id, token):
    try:
        api  = HfApi(token=token)
        info = api.repo_info(repo_id=repo_id, repo_type='dataset')
        size = getattr(info, 'size_on_disk', None)
        if size is None:
            size = sum(getattr(s, 'size', 0) or 0 for s in getattr(info, 'siblings', []))
            if isinstance(size, (int, float)):
                return size / 1024**3
    except Exception:
        pass
    return 0.0


def commit_batch(batch, repo_id, token, wave_num, batch_idx, total_batches):
    api = HfApi(token=token)
    ops = [
        CommitOperationAdd(
            path_in_repo=f'audio/{p.name}',
            path_or_fileobj=str(p),
        )
        for p in batch if p.exists()
    ]
    if not ops:
        return True
    for attempt in range(MAX_RETRIES):
        try:
            api.create_commit(
                repo_id=repo_id,
                repo_type='dataset',
                commit_message=f'wave {wave_num} batch {batch_idx}/{total_batches} — {len(ops)} files',
                operations=ops,
            )
            time.sleep(COMMIT_DELAY)
            return True
        except Exception as e:
            if attempt == MAX_RETRIES - 1:
                print(f'  [commit] wave {wave_num} batch {batch_idx} FAILED: {e}')
                return False
            wait = min(2 ** attempt, 120)
            print(f'  [commit] attempt {attempt+1}/{MAX_RETRIES} failed: {e} — retry in {wait}s')
            time.sleep(wait)
    return False


def flush_wave(wave_files, wave_num, repo_id, token):
    existing   = [p for p in wave_files if p.exists()]
    total_size = sum(p.stat().st_size for p in existing)
    size_mb    = total_size / 1024 / 1024

    print(f'\n[wave {wave_num}] {len(existing)} files ({size_mb:.1f} MB) → {repo_id}')

    batches        = [existing[i:i + BATCH_SIZE] for i in range(0, len(existing), BATCH_SIZE)]
    wave_committed = 0

    for idx, batch in enumerate(batches):
        ok = commit_batch(batch, repo_id, token, wave_num, idx + 1, len(batches))
        if ok:
            wave_committed += len(batch)
            with cp_lock:
                for p in batch:
                    uploaded_set.add(p.stem)
                    state['uploaded_ids'].append(p.stem)
                state['stats']['uploaded'] += len(batch)
            print(f'  batch {idx+1}/{len(batches)} OK — {len(batch)} files')
        else:
            with cp_lock:
                for p in batch:
                    state['failed_ids'].append(p.stem)
                state['stats']['failed'] += len(batch)

    with cp_lock:
        state['stats']['waves_committed'] += 1

    for p in existing:
        p.unlink(missing_ok=True)

    save_checkpoint(state, upload=True)
    print(f'[wave {wave_num}] done — {wave_committed}/{len(existing)} committed, buffer cleared\n')
    return wave_committed


all_wav_files = sorted(CLEAN_DIR.glob('*.wav'))
pending       = [p for p in all_wav_files if p.stem not in uploaded_set]

total_pending_gb = sum(p.stat().st_size for p in pending) / 1024**3
print(f'[upload] {len(all_wav_files)} total WAVs, {len(uploaded_set)} already uploaded, {len(pending)} pending')
print(f'[upload] pending size: {total_pending_gb:.2f} GB')

In [ ]:
HF_REPO_SIZE_LIMIT_GB = 290.0

wave_buf       = []
wave_bytes     = 0
wave_num       = state['stats']['waves_committed'] + 1
current_repo   = STAGE0_REPO
current_token  = HF_TOKEN

for file_idx, wav_path in enumerate(pending):
    file_size = wav_path.stat().st_size

    repo_size_gb = get_repo_size_gb(current_repo, current_token)
    if repo_size_gb >= HF_REPO_SIZE_LIMIT_GB:
        print(f'[overflow] {current_repo} at {repo_size_gb:.1f}GB — switching to overflow repo')
        current_repo  = OVERFLOW_REPO
        current_token = OVERFLOW_TOKEN
        with cp_lock:
            state['stats']['overflow_files'] += 1

    wave_buf.append(wav_path)
    wave_bytes += file_size

    if wave_bytes >= WAVE_SIZE_BYTES:
        flush_wave(wave_buf, wave_num, current_repo, current_token)
        wave_buf   = []
        wave_bytes = 0
        wave_num  += 1

if wave_buf:
    flush_wave(wave_buf, wave_num, current_repo, current_token)

print(f'[upload] all WAV files processed')

In [ ]:
if METADATA_PATH.exists() and not state['metadata_uploaded']:
    print('[metadata] uploading metadata.jsonl...')
    for attempt in range(MAX_RETRIES):
        try:
            HF_API.upload_file(
                path_or_fileobj=str(METADATA_PATH),
                path_in_repo='metadata.jsonl',
                repo_id=STAGE0_REPO,
                repo_type='dataset',
                commit_message='metadata.jsonl — full corpus metadata',
            )
            state['metadata_uploaded'] = True
            save_checkpoint(state, upload=False)
            print('[metadata] uploaded successfully')
            break
        except Exception as e:
            wait = min(2 ** attempt, 120)
            print(f'[metadata] upload attempt {attempt+1}/{MAX_RETRIES} failed: {e} — retry in {wait}s')
            time.sleep(wait)
elif state['metadata_uploaded']:
    print('[metadata] already uploaded, skipping')
else:
    print('[metadata] metadata.jsonl not found — run p1d first')

In [ ]:
if not state['stats_uploaded']:
    total_records  = 0
    total_dur      = 0.0
    codec_ready    = 0
    ce_ready       = 0
    code_switch    = 0
    demucs_count   = 0

    if METADATA_PATH.exists():
        with open(METADATA_PATH, encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                r = json.loads(line)
                total_records += 1
                total_dur     += r.get('audio', {}).get('duration_sec', 0)
                if r.get('quality', {}).get('usable_for_codec'):
                    codec_ready += 1
                if r.get('quality', {}).get('usable_for_ce'):
                    ce_ready += 1
                if r.get('transcript', {}).get('contains_code_switch'):
                    code_switch += 1
                if r.get('audio', {}).get('demucs_applied'):
                    demucs_count += 1

    stats = {
        'total_segments':    total_records,
        'total_duration_h':  round(total_dur / 3600, 2),
        'usable_for_codec':  codec_ready,
        'usable_for_ce':     ce_ready,
        'code_switched':     code_switch,
        'demucs_applied':    demucs_count,
        'uploaded_files':    state['stats']['uploaded'],
        'failed_files':      state['stats']['failed'],
        'waves_committed':   state['stats']['waves_committed'],
        'overflow_files':    state['stats']['overflow_files'],
        'generated_at':      datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'),
    }

    for attempt in range(MAX_RETRIES):
        try:
            HF_API.upload_file(
                path_or_fileobj=json.dumps(stats, indent=2).encode(),
                path_in_repo='stats.json',
                repo_id=STAGE0_REPO,
                repo_type='dataset',
                commit_message='stats.json — corpus statistics',
            )
            state['stats_uploaded'] = True
            save_checkpoint(state, upload=True)
            print('[stats] uploaded successfully')
            break
        except Exception as e:
            wait = min(2 ** attempt, 120)
            print(f'[stats] upload attempt {attempt+1}/{MAX_RETRIES} failed: {e} — retry in {wait}s')
            time.sleep(wait)

    print('\n[corpus stats]')
    for k, v in stats.items():
        print(f'  {k:<25}: {v}')

In [ ]:
print('\n[p1e] final summary')
print(f'  uploaded WAVs     : {state["stats"]["uploaded"]}')
print(f'  failed WAVs       : {state["stats"]["failed"]}')
print(f'  waves committed   : {state["stats"]["waves_committed"]}')
print(f'  overflow files    : {state["stats"]["overflow_files"]}')
print(f'  metadata uploaded : {state["metadata_uploaded"]}')
print(f'  stats uploaded    : {state["stats_uploaded"]}')

if state['failed_ids']:
    failed_log = WORK_DIR / 'failed_uploads.txt'
    with open(failed_log, 'w') as f:
        for seg_id in state['failed_ids']:
            f.write(seg_id + '\n')
    print(f'\n  failed log: {failed_log} ({len(state["failed_ids"])} entries)')
    print('  re-run this notebook to retry failed uploads automatically')

print(f'\n[done] stage0 repo: https://huggingface.co/datasets/{STAGE0_REPO}')
print('[done] pipeline 1 complete — ready for pipeline 2 (p2a_label.ipynb)')